This notebook uses the Generators and Scheduled Loads list retrieved from the most recent NEM Registrations and Exemptions list (retrieved using the NEMOSIS package) as a master list of generators and DUIDs. It currently ignores the ancilliary services list.

The master list is cross checked with the national map CSV of all generators (downloaded from the website in Nov 2024) to assign longitude and latitude coordinates for each generator.

The remaining stations are retrieved from the Find Place Maps API.

The remainder were filled in manually.

In [1]:
import pandas as pd, numpy, requests
from nemosis import static_table
from pathlib import Path

ModuleNotFoundError: No module named 'nemosis'

Below are the locations for file access/filesaving. The contents of these folders are not uploaded to Github.

In [ ]:
workingDir = Path().absolute()
print(f"{workingDir}")

# Read
nmap_loc = '/g/data/ng72/ms5578/ID_HW_BARRA/data/raw'

#Write
output = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'

Here we read and extract the national map power station locations, as well as the Generators and Scheduled loads tab of the NEM Registrations and Exemptions spreadsheet, excluding the loads and bidirectional units. 

Then the National Map and Scheduled Loads tables are left joined. We consider the AEMO data as the complete set and fill gaps where possible.

In [ ]:
cols = {'Site Name':'Station Name',
         'Owner':'Participant',
         'Technology Type':'Technology Type - Primary',
         'Fuel Type':'Fuel Source - Primary',
         'Nameplate Capacity (MW)': 'Reg Cap (MW)',
         'Dispatch Type':'Classification',
         'DUID':'DUID'}

NEM_gen = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/gen_info.csv')
NEM_gen.drop(NEM_gen.filter(regex="Unname"),axis=1, inplace=True)
NEM_gen = NEM_gen[list(cols.keys())].rename(columns=cols)

In [ ]:
nmap_index = pd.read_csv(f"{nmap_loc}/nmap.csv")

# PROGRESS: matching the columns and clearing out duplicates in only one table or other because of arbitrary differences in columns
table = pd.merge(NEM_gen,nmap_index,
                 how="left",
                 on=["DUID"],
                 indicator=True,
                 suffixes=("","_nmap"))

address_list = table.loc[table['Lat'].isnull()]['Station Name']

Now we try to fill in the gaps using the Google Maps Find Place API.

In [ ]:
def get_place_info(address, api_key):
# Base URL
  base_url = "https://maps.googleapis.com/maps/api/place/textsearch/json"
# Parameters in a dictionary
  params = {
   "query": address,
   "inputtype": "textquery",
   "fields": "formatted_address,name,geometry,place_id",
   "key": api_key,
  }
# Send request and capture response
  response = requests.get(base_url, params=params)
# Check if the request was successful
  if response.status_code == 200:
    return response.json()
  else:
    return None

Here we print the station we're looking up and the formatted address returned by Google (if a match was found) so I can easily verify the locations.

In [ ]:
for address in address_list:
  api_key = "REDACTED_API_KEY"
  place_info = get_place_info(address, api_key)
  if place_info is not None:
    print(address)
    if place_info['status'] != 'ZERO_RESULTS':
      print(place_info["results"][0]["formatted_address"])
      table.loc[table['Station Name'] == address, "Lat"] = place_info["results"][0]["geometry"]["location"]["lat"]
      table.loc[table['Station Name'] == address, "Lon"] = place_info["results"][0]["geometry"]["location"]["lng"]
    else:
      print('Zero results.')
  else:
    print("Failed to get a response from Google Places API")

Below are the entries I had to manually fill due to the Find Place search returning no results. This was done here, rather than Excel, due to some column values being incorrectly read as dates. Where an exact location was unavailable, I used the town of the power station.

As the Basslink HVDC is not a generator, its location has been left as NaN.

In [ ]:
table.loc[table['Station Name'] == "Bouldercombe Battery Project", ["Lat","Lon"]] = -23.57054, 150.46951
table.loc[table['Station Name'] == "Broken Hill Battery Energy Storage System", ["Lat","Lon"]] = -31.95995500172762, 141.45556447263974
table.loc[table['Station Name'] == "Christies Beach Wastewater Treatment Plant", ["Lat","Lon"]] = -35.12587478747303, 138.47142547116428
table.loc[table['Station Name'] == "Columboola Solar Farm", ["Lat","Lon"]] = -26.66805960796126, 150.32432238093762
table.loc[table['Station Name'] == "Goyder South Wind Farm 1A", ["Lat","Lon"]] = -33.75, 138.92
table.loc[table['Station Name'] == "Goyder South Wind Farm 1B", ["Lat","Lon"]] = -33.75, 138.92
table.loc[table['Station Name'] == "Happy Valley Water Treatment Plant", ["Lat","Lon"]] = -35.07276474208724, 138.570830349246
table.loc[table['Station Name'] == "Lake Bonney Wind Farm Stage 1", ["Lat","Lon"]] = -37.73772964339552, 140.38687627116428
table.loc[table['Station Name'] == "Mannum - Adelaide Pipeline Pumping Station No 3, PV Units 1-6", ["Lat","Lon"]] = -34.83893733218697, 139.13325694146798
table.loc[table['Station Name'] == "Mugga Lane Landfill Gas To Energy Facility", ["Lat","Lon"]] = -35.384997248702916, 149.13793163637578
table.loc[table['Station Name'] == "Murra Warra Wind Farm Stage 2", ["Lat","Lon"]] = -36.42931970054318, 142.3193529864758
table.loc[table['Station Name'] == "Queanbeyan BESS", ["Lat","Lon"]] = -35.336691822184726, 149.21569374239218
table.loc[table['Station Name'] == "Tailem Bend 2 Hybrid Renewable Power Station", ["Lat","Lon"]] = -35.25195170827054, 139.45636616412477
table.loc[table['Station Name'] == "Tungatinah Power Station", ["Lat","Lon"]] = -42.29662855198331, 146.45663948650713
table.loc[table['Station Name'] == "Walla Walla Solar Farm 1", ["Lat","Lon"]] = -35.76323647254411, 146.89826056491617
table.loc[table['Station Name'] == "Walla Walla Solar Farm 2", ["Lat","Lon"]] = -35.76323647254411, 146.89826056491617
table.loc[table['Station Name'] == "Woolooga Solar Farm", ["Lat","Lon"]] = -26.069894261418906, 152.44690338913858
table.loc[table['Station Name'] == "Cathedral Rocks", ["Lat","Lon"]] = -34.864972145661355, 135.59663440865043
table.loc[table['Station Name'] == "Metz Solar Farm", ["Lat","Lon"]] = -30.503974792913095, 151.65395252749553
table.loc[table['Station Name'] == "New England Solar Farm", ["Lat","Lon"]] = -30.620850420860823, 151.61851916661263
table.loc[table['Station Name'] == "Hastings Generation Site", ["Lat","Lon"]] = -38.307744423182264, 145.18282955259676
table.loc[table['Station Name'] == "Hunter Economic Zone", ["Lat","Lon"]] = -32.167037319746925, 149.02657039398943
table.loc[table['Station Name'] == "Stubbo Solar Farm 1", ["Lat","Lon"]] = -32.26230412325072, 149.58908943685117





In [ ]:
table.loc[table['Lat'].isnull()]['Station Name']

In [ ]:
# table = table.drop(labels=['_merge'], axis="columns")
table.columns = (
    table.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
    )
table = table.rename(columns={'duid': 'DUID'})

table.to_csv(f"{output}/gen_info.csv", na_rep='',index=False)

Below we check for any rows in table that don't have results. Basslink is an exception as it's not a generator of interest, being an HVDC cable between states.

In [ ]:
np.unique(table['fuel_source_primary'])